# Experiment with Genomics of Drug Sensitivity in Cancer Dataset

This Jupyter Notebook implements the experiments and methods for the Genomics of Drug Sensitivity in Cancer (GDSC) dataset analysis. It provides all necessary functions to load, preprocess, and analyze the GDSC data, as well as to run multiple hypothesis testing procedures and evaluate their performance.

## Key Features

- **Data Loading and Preprocessing:** Functions are provided to load the GDSC drug response data and genomic features, filter and preprocess them for downstream analysis.
- **Statistical Methods:** Implements various multiple testing correction methods, including Benjamini-Hochberg (BH), pooled BH, and a synthetic-powered BH procedure.
- **Experiment Pipeline:** The `run_experiment` function orchestrates the entire experiment, including data splitting, p-value computation, hypothesis testing, and result aggregation.
- **Evaluation:** The notebook supports evaluation of methods using ground truth scores, which can be generated and saved for reproducibility (provided).

## Usage

All required packages are listed below. All necessary data files should be present in the specified folders.

To run an experiment, simply call the `run_experiment` function with the desired parameters. See example in the end of this notebook.

### Required packages:
- numpy
- pandas
- scipy
- statsmodels
- tqdm
- seaborn
- matplotlib

In [ ]:
import numpy as np
import random
import os
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
from utils_plot import plot4paper


In [10]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)

def get_run_description(**kwargs):
    run_name = ''
    for key, value in kwargs.items():
        run_name += f"{key}_{value}_"
    return run_name.rstrip('_')


## Data handlers

In [11]:
def load_data(data_dir_path,
              drug_response_filename="GDSC2_fitted_dose_response_24Jul22.csv"):
    """
    Load GDSC fitted dose response.
    Saves a full processed file with all columns but filtered rows.
    Returns only relevant columns for simulation.
    
    Returns:
        df: DataFrame with columns:
            COSMIC_ID, DRUG_NAME, LN_IC50, PUTATIVE_TARGET, cancer_type
    """
    processed_file = f"{data_dir_path}/processed/gdsc_processed.csv"

    # If processed file exists, load it
    if os.path.exists(processed_file):
        print("Loading preprocessed data...")
        df_full = pd.read_csv(processed_file, index_col=0)
    else:
        # Ensure directories exist
        os.makedirs(data_dir_path, exist_ok=True)
        os.makedirs(f"{data_dir_path}/processed/", exist_ok=True)

        # Load raw CSV
        raw_file = f"{data_dir_path}/{drug_response_filename}"
        if not os.path.exists(raw_file):
            raise ValueError(f'Error: data not found, the following path does not exist - {raw_file}.')

        df_full = pd.read_csv(raw_file)

        # Drop invalid rows (unclassified / other / missing cancer types)
        invalid_cancers = ["UNCLASSIFIED", "OTHER"]
        df_full = df_full[~df_full["TCGA_DESC"].isin(invalid_cancers)].dropna(subset=["TCGA_DESC"])


        # Save the full processed file with all columns
        df_full.to_csv(processed_file)
        print(f"Processed data saved to {processed_file}")

    # Return only relevant columns for simulation
    df_sim = df_full[["COSMIC_ID", "DRUG_NAME", "LN_IC50", "PUTATIVE_TARGET", "cancer_type"]].copy()
    return df_sim


def split_real_auxiliary(df, target_cancer=['BRCA']):
    df_real_dict = {}
    for tissue in target_cancer:
        df_real_dict[tissue] = df[df["cancer_type"] == tissue].copy()
    df_aux = df[~df["cancer_type"].isin(target_cancer)].copy()
    return df_real_dict, df_aux


def subsample_ic50_matrix(
    gdsc2_drug_response_df,
    frac_data=1.0,
    random_state=0,
):
    """
    Input:
        gdsc2_drug_response_df with columns:
            - COSMIC_ID
            - DRUG_NAME
            - LN_IC50

    Output:
        IC50 matrix (rows: COSMIC_ID, columns: DRUG_NAME)
        suitable for GDSCTools ANOVA
    """
    rng = np.random.default_rng(random_state)

    df = gdsc2_drug_response_df[
        ["COSMIC_ID", "DRUG_NAME", "LN_IC50"]
    ].dropna()

    sampled = []

    for drug, df_d in df.groupby("DRUG_NAME"):
        ids = df_d["COSMIC_ID"].unique()
        n = int(len(ids) * frac_data)
        sampled_ids = rng.choice(ids, n, replace=False)

        sampled.append(
            df_d[df_d["COSMIC_ID"].isin(sampled_ids)]
        )

    df_sub = pd.concat(sampled, ignore_index=True)

    ic50_matrix = df_sub.pivot_table(
        index="COSMIC_ID",
        columns="DRUG_NAME",
        values="LN_IC50",
        aggfunc="mean"
    )

    return ic50_matrix



def compute_pvalues(ic50_matrix, features_df, min_per_group=3, selected_hypotheses=None, tissue=None):
    """
    Compute ANOVA p-values per (drug, feature) pair, aligned with GDSCTools.

    Parameters
    ----------
    ic50_matrix : pd.DataFrame
        Rows: COSMIC_ID, Columns: DRUG_NAME, Values: IC50 or LN_IC50
    features_df : pd.DataFrame
        Rows: COSMIC_ID, Columns: features, binary 0/1
    min_per_group : int
        Minimum number of cell lines in each feature group to perform ANOVA

    Returns
    -------
    pd.DataFrame
        Columns: DRUG, FEATURE, PVAL
    """
    results = []

    # Align indices
    shared_ids = ic50_matrix.index.intersection(features_df.index)
    ic50_matrix = ic50_matrix.loc[shared_ids]
    features_df = features_df.loc[shared_ids]

    if selected_hypotheses is None:
        raise ValueError('Must pass selected hypotheses.')
    for _, row in selected_hypotheses.iterrows():
        tissue_ = row['tissue']
        if tissue is not None and tissue != tissue_:
            continue
        feature = row['feature']
        drug = row['drug']
        ic50_drug = ic50_matrix[drug]

        vals = features_df[feature]

        # Drop missing IC50 or feature values
        mask = ic50_drug.notna() & vals.notna()
        group1 = ic50_drug[vals == 1][mask]
        group0 = ic50_drug[vals == 0][mask]

        if len(group1) >= min_per_group and len(group0) >= min_per_group:
            stat, pval = ttest_ind(group1, group0, equal_var=True)
            results.append({"TISSUE": tissue, "DRUG": drug, "FEATURE": feature, "PVAL": pval})
        else:
            raise ValueError(f"Not enough cell lines for testing -- (drug={drug}, feature={feature}) - len(group1)={len(group1)}, len(group0)={len(group0)}")
    return pd.DataFrame(results)
    

def shared_pvalues(df1, df2, df3):
    """
    Returns p-values for hypotheses shared across all three runs.
    """
    return (
        df1
        .merge(df2, on=["TISSUE", "DRUG", "FEATURE"], suffixes=("_run1", "_run2"))
        .merge(df3, on=["TISSUE", "DRUG", "FEATURE"])
        .rename(columns={"PVAL": "PVAL_run3"})
    )



## Methods

In [12]:

def synth_powered_BH(pvalues_real, pvalues_synth_powered, level, epsilon):
    """
    synthetic-powered Benjamini-Hochberg procedure.
    """
    p = np.asarray(pvalues_real)
    p_synth = np.asarray(pvalues_synth_powered)
    m = len(p)

    k_vals = np.arange(1, m + 1)
    adj = (k_vals * epsilon / m)[:, None]  # shape (m, 1)

    # Compute \tilde{p}_{k,j} = min{p_j, max{p^synth_j, p_j - k*epsilon/m}}
    P_tilde = np.minimum(p, np.maximum(p_synth, p - adj))  # shape (m, m)

    # Compute the k-th smallest element for each row (using partition or full sort)
    p_kth = np.partition(P_tilde, k_vals - 1, axis=1)[np.arange(m), k_vals - 1]
    # p_kth = np.sort(P_tilde, axis=1)[np.arange(m), k_vals - 1]

    thresholds = level * k_vals / m
    cond = p_kth <= thresholds

    if np.any(cond):
        k_star = np.max(np.where(cond)[0]) + 1  # +1 for 1-based indexing
        # Recompute \tilde{p} for k_star only
        P_tilde_star = np.minimum(p, np.maximum(p_synth, p - k_star * epsilon / m))
        threshold = np.partition(P_tilde_star, k_star - 1)[k_star - 1]
        rejection_mask = P_tilde_star <= threshold
    else:
        P_tilde_star = []
        k_star, threshold = 0, 0
        rejection_mask = np.zeros(m, dtype=bool)

    return rejection_mask, threshold, P_tilde_star

In [13]:

def run_experiment(
        save_path,
        data_dir_path='./data/GDSC2/',
        features_file_path="./data/GDSC2/features_info/genomic_features_v17.csv",
        target_cancer=['BRCA'],
        selected_hypotheses_path=None,
        gt_scores_path=None,
        compute_gt_scores=False,
        frac_data=1,
        alpha=0.1,
        epsilon=0.1,
        n_runs=10,
        seed=42,
):
    set_seed(seed)
    seed_list = random.sample(range(1, 999999), n_runs*2)
    count_valid_runs = 0
    data = load_data(data_dir_path)
    features_df = pd.read_csv(features_file_path)
    if "COSMIC_ID" in features_df.columns:
        features_df = features_df.set_index("COSMIC_ID")
    df_real_dict, df_aux = split_real_auxiliary(data, target_cancer)
    if selected_hypotheses_path is not None:
        selected_hypotheses = pd.read_csv(selected_hypotheses_path)
    else:
        selected_hypotheses = None
    if gt_scores_path is not None:
        gt_scores = pd.read_csv(gt_scores_path)
    else:
        gt_scores = None
    
    params_dict = {'target_cancer': '-'.join(target_cancer), 
                   'alpha': alpha, 'epsilon': epsilon, 'n_runs': n_runs}
    results = pd.DataFrame()
    for seed_ in tqdm(seed_list):
        set_seed(seed_)
        # Subsample synthetic data
        ic50_matrix_dict = {}
        ic50_matrix_aux = subsample_ic50_matrix(df_aux, random_state=seed_, frac_data=frac_data)
        for tissue in target_cancer:
            # print(f"Subsampling real data for tissue: {tissue}")
            ic50_matrix = subsample_ic50_matrix(df_real_dict[tissue], random_state=seed_, frac_data=frac_data)
            ic50_combined = pd.concat([ic50_matrix, ic50_matrix_aux], axis=0)
            ic50_matrix_dict[tissue] = (ic50_matrix, ic50_matrix_aux, ic50_combined)
        curr_results = run_single_experiment(
            ic50_matrix_dict, features_df,
            selected_hypotheses=selected_hypotheses,
            alpha=alpha,
            epsilon=epsilon,
            seed=seed_,
            gt_scores=gt_scores,
            compute_gt_scores=compute_gt_scores,
            params_dict=params_dict,
        )
        if isinstance(curr_results, str) and curr_results == 'not_valid':
            continue
        count_valid_runs += 1
        results = pd.concat([results, curr_results], ignore_index=True)
        if count_valid_runs >= n_runs:
            break
    print(f"Completed {count_valid_runs} valid runs.")
    if compute_gt_scores:
        gt_scores = (
            results
            .groupby(["TISSUE", "DRUG", "FEATURE"])["rejected"]
            .mean()
            .reset_index()
            .rename(columns={"rejected": "gt_prob"})
        )
        return gt_scores

    # save results
    run_desc = get_run_description(**params_dict)
    save_path_ = os.path.join(save_path, run_desc)
    os.makedirs(save_path_, exist_ok=True)
    results.to_pickle(os.path.join(save_path_, "results.pkl"))
    methods2plot = ['BH_real', 'BH_pooled', 'SynthBH', 'BH_real+e']
    for y in ['n_rejections', 'gt_score', 'extra_rejections', 'extra_gt_score']:
        plot4paper(save_path_, results, x='n_runs', y=y, 
                   hue='Method', methods2plot=methods2plot, alpha=alpha, epsilon=epsilon, remove_x=True)
    return results

def run_single_experiment(
    ic50_matrix_dict, features_df,
    selected_hypotheses=None,
    alpha=0.1,
    epsilon=0.1,
    seed=42,
    gt_scores=None,
    compute_gt_scores=False,
    params_dict=None,
):
    all_df_p_real = []
    all_df_p_aux = []
    all_df_p_pooled = []
    for tissue in ic50_matrix_dict.keys():
        ic50_matrix, ic50_matrix_aux, ic50_combined = ic50_matrix_dict[tissue]
        try:
            # Compute p-values
            df_p_real = compute_pvalues(ic50_matrix, features_df, selected_hypotheses=selected_hypotheses, tissue=tissue)
            df_p_aux = compute_pvalues(ic50_matrix_aux, features_df, selected_hypotheses=selected_hypotheses, tissue=tissue)
            df_p_pooled = compute_pvalues(ic50_combined, features_df, selected_hypotheses=selected_hypotheses, tissue=tissue)
        except Exception as e:
            return 'not_valid'
        all_df_p_real.append(df_p_real)
        all_df_p_aux.append(df_p_aux)
        all_df_p_pooled.append(df_p_pooled)

    df_p_real = pd.concat(all_df_p_real, ignore_index=True)
    df_p_aux = pd.concat(all_df_p_aux, ignore_index=True)
    df_p_pooled = pd.concat(all_df_p_pooled, ignore_index=True)
    # order p-values by (drug, feature)
    shared = shared_pvalues(df_p_real, df_p_aux, df_p_pooled)
    p_real = shared["PVAL_run1"].to_numpy()
    p_aux = shared["PVAL_run2"].to_numpy()
    p_pooled = shared["PVAL_run3"].to_numpy()
    
    reject_real, _, _, _ = multipletests(p_real, alpha=alpha, method="fdr_bh")
    reject_real_e, _, _, _ = multipletests(p_real, alpha=alpha+epsilon, method="fdr_bh")
    reject_aux_real, _, _, _ = multipletests(p_pooled, alpha=alpha, method="fdr_bh")
    reject_aux, _, _, _ = multipletests(p_aux, alpha=alpha, method="fdr_bh")
    reject_pooled, _, P_tilde_star = synth_powered_BH(p_real, p_pooled, alpha, epsilon)

    if compute_gt_scores:
        gt_df = shared[["TISSUE", "DRUG", "FEATURE"]].copy()
        gt_df["rejected"] = reject_real.astype(int)
        return gt_df

    extra_mask = reject_pooled & (~reject_real)

    scores = {
        "BH_synth_gt_score": score_rejections(reject_aux, gt_scores, shared),
        "BH_pooled_gt_score": score_rejections(reject_aux_real, gt_scores, shared),
        "BH_real_gt_score": score_rejections(reject_real, gt_scores, shared),
        "BH_real_e_gt_score": score_rejections(reject_real_e, gt_scores, shared),
        "SynthBH_gt_score": score_rejections(reject_pooled, gt_scores, shared),
        "SynthBH_extra_gt_score": score_rejections(extra_mask, gt_scores, shared),
        "SynthBH_extra_rejections": int(extra_mask.sum()),
    }

    curr_results = pd.DataFrame([{**params_dict, 'Method': 'BH_real', 'n_rejections': reject_real.sum(),
                                  'gt_score': scores.get("BH_real_gt_score", np.nan),},
                                {**params_dict, 'Method': 'BH_real+e', 'n_rejections': reject_real_e.sum(),
                                 'gt_score': scores.get("BH_real_e_gt_score", np.nan),},
                                {**params_dict, 'Method': 'BH_synth', 'n_rejections': reject_aux.sum(),
                                 'gt_score': scores.get("BH_synth_gt_score", np.nan),},
                                {**params_dict, 'Method': 'BH_pooled', 'n_rejections': reject_aux_real.sum(),
                                 'gt_score': scores.get("BH_pooled_gt_score", np.nan),},
                                {**params_dict, 'Method': 'SynthBH', 'n_rejections': reject_pooled.sum(),
                                 'gt_score': scores.get("SynthBH_gt_score", np.nan),
                                 'extra_rejections': scores.get("SynthBH_extra_rejections", np.nan),
                                 'extra_gt_score': scores.get("SynthBH_extra_gt_score", np.nan),},
                                ])
    return curr_results


def score_rejections(reject_mask, gt_scores, shared):
    if gt_scores is None or reject_mask.sum() == 0:
        return np.nan
    rejected_hypos = shared.loc[reject_mask, ["TISSUE", "FEATURE", "DRUG"]]
    merged = rejected_hypos.merge(
        gt_scores,
        on=["TISSUE", "FEATURE", "DRUG"],
        how="left"
    )
    return merged["gt_prob"].mean()


## Run Experiments

#### Create Ground-Truth Scores
No need to run, file already exsits.

In [ ]:
## save ground truth scores
gt_scores = run_experiment(
        save_path='./results/',
        data_dir_path='./data/GDSC2/',
        features_file_path="./data/GDSC2/features_info/genomic_features_v17.csv",
        target_cancer=['LUAD', 'BRCA'],
        selected_hypotheses_path="./selected_hypotheses.csv",
        compute_gt_scores=True,
        frac_data=0.8,
        alpha=0.1,
        epsilon=0.05,
        n_runs=10,
        seed=42,
)
gt_scores.to_csv("./gt_scores_for_selected_hypotheses.csv", index=False)


#### Run specific experiment

In [15]:

results = run_experiment(
        save_path='./results/',
        data_dir_path='./data/GDSC2/',
        features_file_path="./data/GDSC2/features_info/genomic_features_v17.csv",
        target_cancer=['LUAD', 'BRCA'],
        selected_hypotheses_path="./selected_hypotheses.csv",
        gt_scores_path="./gt_scores_for_selected_hypotheses.csv",
        frac_data=0.5,
        alpha=0.1,
        epsilon=0.1,
        n_runs=10,
        seed=42,
)


Loading preprocessed data...


 90%|█████████ | 18/20 [00:09<00:01,  1.92it/s]


Completed 10 valid runs.


In [16]:

results = run_experiment(
        save_path='./results/',
        data_dir_path='./data/GDSC2/',
        features_file_path="./data/GDSC2/features_info/genomic_features_v17.csv",
        target_cancer=['LUAD', 'BRCA'],
        selected_hypotheses_path="./selected_hypotheses.csv",
        gt_scores_path="./gt_scores_for_selected_hypotheses.csv",
        frac_data=0.5,
        alpha=0.1,
        epsilon=0.05,
        n_runs=10,
        seed=42,
)


Loading preprocessed data...


 90%|█████████ | 18/20 [00:09<00:01,  1.92it/s]


Completed 10 valid runs.
